# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Scoring / Ranking

**Why:** The operational objective is not a hard binary decision, but predicting continuous expected impact to score and rank entities (e.g., pages, queries, or content updates). Ordering targets relative to one another enables downstream automated workflows and operations to systematically prioritize high-value targets.

In [ ]:
# Setup and configuration
import pandas as pd
import numpy as np

# Define framing metadata
TASK_TYPE = "Scoring / Ranking"
print(f"Configured Task Type: {TASK_TYPE}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy Definition:** `observed_engagement_score` (Continuous Proxy)

**Label Source:** Calculated from observed historical metric logs (e.g., click-through rate, dwell time, or ranking position delta) over a fixed 30-day observation window following an update. It represents a measured, empirical outcome rather than a synthetic or rule-based label.

In [ ]:
# Target formulation definition
TARGET_COL = "target_proxy_score"
OBSERVATION_WINDOW_DAYS = 30

print(f"Target Column: {TARGET_COL} (Window: {OBSERVATION_WINDOW_DAYS} days)")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** Spearman's Rank Correlation ($ho$) and Mean Absolute Error (MAE).

**Defense:** Since downstream actions rely on proper prioritization, rank correlation evaluates how effectively the model preserves relative item ordering against observed outcomes. MAE bounds prediction error magnitude without over-penalizing non-pathological outliers as MSE does. Achieving $ho > 0.35$ against baseline confirms a meaningful directional signal for decision-support.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

def evaluate_baseline(y_true, y_pred):
    """Calculates directional rank agreement and mean absolute error."""
    rho, p_val = spearmanr(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {"spearman_rho": rho, "mae": mae, "p_value": p_val}

# Verification test
print(evaluate_baseline([0.1, 0.4, 0.8], [0.2, 0.3, 0.9]))

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = One entity snapshot at a specific point in time.

**Data Leakage Constraint:** Feature attributes are calculated exclusively from historical data windows preceding the snapshot timestamp.

In [ ]:
# Load starter data slice or verify unit of analysis schema
sample_data = {
    "entity_id": ["URL_001", "URL_002", "URL_003"],
    "timestamp": ["2026-06-01", "2026-06-01", "2026-06-01"],
    "feature_historical_views": [1200, 4500, 310],
    "feature_content_length": [850, 2100, 400],
    "target_proxy_score": [0.72, 0.89, 0.15]
}

df_unit = pd.DataFrame(sample_data)

print(f"Unit of analysis shape: {df_unit.shape}")
print("One row = one entity instance at a point in time.")
df_unit.head()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML:** Deterministic `IF/THEN` heuristics fail due to non-linear feature interactions, high variance across dynamic categories, and shifting baseline distributions over time. Fixed thresholds (e.g., `IF content_length > 1000 AND views > 1000`) introduce brittle boundary conditions and fail to weigh continuous, multi-dimensional signals effectively. Machine learning dynamically models complex non-linear relationships to produce a robust continuous score.

In [ ]:
# Rule baseline example demonstrating fragility vs continuous ML target
def rule_baseline(df):
    """Rigid heuristic rule threshold."""
    return (df["feature_historical_views"] > 1000) & (df["feature_content_length"] > 1000)

df_unit["rule_pred"] = rule_baseline(df_unit).astype(int)
df_unit[["entity_id", "target_proxy_score", "rule_pred"]]

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.